# BT01: Mô hình ngôn ngữ Bigram âm tiết cho tiếng Việt

**Yêu cầu:**
1. Xây dựng mô hình ngôn ngữ **bigram** cho tiếng Việt mức âm tiết.
2. Tính xác suất của câu **"Hôm nay trời đẹp lắm"**
3. Sinh ra một số câu từ mô hình đã xây dựng

> **Corpus:** Tải tự động từ Wikipedia tiếng Việt thông qua Wikipedia API.  
> **Ghi chú:** Trong tiếng Việt, mỗi âm tiết cách nhau bởi dấu cách → tokenize = tách theo khoảng trắng.

In [1]:
# ─── Cài đặt thư viện cần thiết ───────────────────────────────────────────────
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'tqdm', '--quiet'])

0

In [2]:
# ─── Import ───────────────────────────────────────────────────────────────────
import re
import math
import random
import json
import time
import requests
from collections import defaultdict, Counter
from tqdm import tqdm

print('✓ Import thành công')

✓ Import thành công


In [13]:
from pathlib import Path

INPUT_FILE = Path('input.txt')
OUTPUT_FILE = Path('output.txt')
CORPUS_FILE = Path('corpus_vi_wiki.txt')

def read_input_sentence(input_path: Path, default_sentence: str) -> str:
    """Đọc câu từ file input; nếu không có thì dùng câu mặc định."""
    if not input_path.exists():
        return default_sentence

    content = input_path.read_text(encoding='utf-8').strip()
    return content if content else default_sentence

def write_output_text(output_path: Path, text: str) -> None:
    """Ghi kết quả ra file output (UTF-8)."""
    output_path.write_text(text, encoding='utf-8')

print(f'Input file : {INPUT_FILE.resolve()}')
print(f'Output file: {OUTPUT_FILE.resolve()}')
print(f'Corpus file: {CORPUS_FILE.resolve()}')

Input file : E:\Cao học\Xử lý ngôn ngữ tự nhiên\input.txt
Output file: E:\Cao học\Xử lý ngôn ngữ tự nhiên\output.txt
Corpus file: E:\Cao học\Xử lý ngôn ngữ tự nhiên\corpus_vi_wiki.txt


## 0. Cấu hình file input/output

## 1. Tải corpus tiếng Việt từ Wikipedia

In [14]:
WIKI_API = 'https://vi.wikipedia.org/w/api.php'
HEADERS  = {'User-Agent': 'VietnameseBigramLM/1.0 (student research)'}

def get_random_titles(n: int = 200) -> list[str]:
    """Lấy n tiêu đề bài viết ngẫu nhiên từ Wikipedia tiếng Việt."""
    titles = []
    batch  = 50  # API giới hạn 500/lần; dùng 50 cho an toàn
    for _ in range(math.ceil(n / batch)):
        params = {
            'action': 'query', 'format': 'json',
            'list': 'random', 'rnnamespace': 0, 'rnlimit': batch
        }
        r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=15)
        r.raise_for_status()
        titles += [p['title'] for p in r.json()['query']['random']]
        time.sleep(0.1)
    return titles[:n]


def fetch_article(title: str) -> str:
    """Lấy nội dung plain-text của một bài Wikipedia."""
    params = {
        'action': 'query', 'format': 'json',
        'titles': title, 'prop': 'extracts',
        'explaintext': True, 'exsectionformat': 'plain'
    }
    r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=15)
    r.raise_for_status()
    pages = r.json()['query']['pages']
    for page in pages.values():
        return page.get('extract', '')
    return ''


def download_corpus(n_articles: int = 200, corpus_path: Path | None = None) -> str:
    """Tải n_articles bài Wikipedia và nối thành một chuỗi."""
    print(f'Đang lấy {n_articles} tiêu đề ngẫu nhiên...')
    titles = get_random_titles(n_articles)

    texts = []
    print('Đang tải nội dung từng bài...')
    for title in tqdm(titles):
        try:
            text = fetch_article(title)
            if text:
                texts.append(text)
        except Exception:
            pass
        time.sleep(0.05)

    corpus = '\n'.join(texts)
    print(f'\n✓ Tải xong: {len(texts)} bài, {len(corpus):,} ký tự')

    if corpus_path is not None:
        corpus_path.write_text(corpus, encoding='utf-8')
        print(f'✓ Đã lưu corpus vào file: {corpus_path.resolve()}')

    return corpus


raw_corpus = download_corpus(n_articles=200, corpus_path=CORPUS_FILE)

Đang lấy 200 tiêu đề ngẫu nhiên...
Đang tải nội dung từng bài...


100%|██████████| 200/200 [03:08<00:00,  1.06it/s]


✓ Tải xong: 200 bài, 175,210 ký tự
✓ Đã lưu corpus vào file: E:\Cao học\Xử lý ngôn ngữ tự nhiên\corpus_vi_wiki.txt


## 2. Tiền xử lý – Trích xuất câu và token hoá âm tiết

In [5]:
# Ký hiệu đặc biệt
BOS = '<s>'   # Begin-of-sentence
EOS = '</s>'  # End-of-sentence


def clean_text(text: str) -> str:
    """Loại bỏ ký tự thừa, chuẩn hoá khoảng trắng."""
    # Bỏ dấu ngoặc, số thứ tự section
    text = re.sub(r'\(.*?\)', ' ', text)          # bỏ ngoặc đơn
    text = re.sub(r'\[.*?\]', ' ', text)          # bỏ ngoặc vuông
    text = re.sub(r'==+.*?==+', ' ', text)        # bỏ tiêu đề section
    text = re.sub(r'http\S+', ' ', text)          # bỏ URL
    text = re.sub(r'[^\w\s\.,!?;:\u00C0-\u024F\u1E00-\u1EFF]', ' ', text)  # giữ chữ Latin + tiếng Việt
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def split_sentences(text: str) -> list[str]:
    """Tách văn bản thành câu."""
    # Tách tại dấu .!? hoặc xuống dòng
    sents = re.split(r'(?<=[.!?])\s+|\n+', text)
    sents = [s.strip() for s in sents if len(s.strip()) > 5]
    return sents


def tokenize_syllables(sentence: str) -> list[str]:
    """
    Trong tiếng Việt, mỗi âm tiết là một 'từ' cách bởi dấu cách.
    Hàm này chuẩn hoá chữ thường và tách theo khoảng trắng.
    """
    # Bỏ dấu câu ở đầu/cuối mỗi token, chuyển về chữ thường
    tokens = [
        re.sub(r'^[^\w\u00C0-\u024F\u1E00-\u1EFF]+|[^\w\u00C0-\u024F\u1E00-\u1EFF]+$', '', tok).lower()
        for tok in sentence.split()
    ]
    return [t for t in tokens if t]  # bỏ token rỗng


def build_sentences(raw: str) -> list[list[str]]:
    """Trả về list các câu đã tokenise, thêm BOS/EOS."""
    cleaned   = clean_text(raw)
    raw_sents = split_sentences(cleaned)
    sentences = []
    for s in raw_sents:
        tokens = tokenize_syllables(s)
        if len(tokens) >= 2:               # bỏ câu quá ngắn
            sentences.append([BOS] + tokens + [EOS])
    return sentences


sentences = build_sentences(raw_corpus)
print(f'Số câu sau tiền xử lý : {len(sentences):,}')
print(f'Ví dụ câu đầu tiên    : {sentences[0]}')

Số câu sau tiền xử lý : 3,996
Ví dụ câu đầu tiên    : ['<s>', 'synechanthus', 'warscewiczianus', 'là', 'loài', 'thực', 'vật', 'có', 'hoa', 'thuộc', 'họ', 'arecaceae', '</s>']


## 3. Xây dựng mô hình Bigram

In [6]:
class BigramLanguageModel:
    """
    Mô hình ngôn ngữ Bigram mức âm tiết cho tiếng Việt.
    Sử dụng Laplace (add-1) smoothing để xử lý n-gram chưa gặp.
    """

    def __init__(self):
        self.unigram_counts: Counter       = Counter()
        self.bigram_counts: defaultdict    = defaultdict(Counter)
        self.vocabulary: set               = set()
        self.vocab_size: int               = 0

    # ── Huấn luyện ──────────────────────────────────────────────────────────
    def train(self, sentences: list[list[str]]) -> None:
        """Đếm unigram và bigram từ tập câu."""
        for sent in sentences:
            for tok in sent:
                self.unigram_counts[tok] += 1
                self.vocabulary.add(tok)
            for w1, w2 in zip(sent[:-1], sent[1:]):
                self.bigram_counts[w1][w2] += 1

        self.vocab_size = len(self.vocabulary)
        total_tokens    = sum(self.unigram_counts.values())
        print(f'✓ Huấn luyện hoàn tất')
        print(f'  Kích thước từ vựng (V) : {self.vocab_size:,}')
        print(f'  Tổng token             : {total_tokens:,}')
        print(f'  Số bigram khác nhau    : {sum(len(v) for v in self.bigram_counts.values()):,}')

    # ── Xác suất bigram có Laplace smoothing ────────────────────────────────
    def bigram_prob(self, w1: str, w2: str) -> float:
        """
        P(w2 | w1) = (C(w1,w2) + 1) / (C(w1) + V)
        """
        count_w1_w2 = self.bigram_counts[w1][w2]
        count_w1    = self.unigram_counts[w1]
        return (count_w1_w2 + 1) / (count_w1 + self.vocab_size)

    # ── Xác suất câu ────────────────────────────────────────────────────────
    def sentence_probability(self, sentence: str, log: bool = False):
        """
        Tính P(câu) = ∏ P(wᵢ | wᵢ₋₁)  (với BOS/EOS)
        Trả về (prob, log_prob)
        """
        tokens   = [BOS] + tokenize_syllables(sentence) + [EOS]
        log_prob = 0.0
        details  = []
        for w1, w2 in zip(tokens[:-1], tokens[1:]):
            p = self.bigram_prob(w1, w2)
            log_prob += math.log(p)
            details.append((w1, w2, p))
        return math.exp(log_prob), log_prob, details

    # ── Sinh câu ────────────────────────────────────────────────────────────
    def generate_sentence(self,
                          seed: str | None = None,
                          max_len: int = 20,
                          temperature: float = 1.0) -> str:
        """
        Sinh một câu mới bằng cách lấy mẫu theo phân phối bigram.
        temperature < 1: thiên về từ phổ biến; > 1: đa dạng hơn.
        """
        current = BOS if seed is None else seed.lower()
        tokens  = [] if seed is None else [seed.lower()]

        for _ in range(max_len):
            candidates = self.bigram_counts.get(current, {})
            if not candidates:
                break

            words  = list(candidates.keys())
            counts = [candidates[w] ** (1.0 / temperature) for w in words]
            total  = sum(counts)
            probs  = [c / total for c in counts]

            current = random.choices(words, weights=probs, k=1)[0]
            if current == EOS:
                break
            tokens.append(current)

        return ' '.join(tokens)


# ── Khởi tạo và huấn luyện ──────────────────────────────────────────────────
model = BigramLanguageModel()
model.train(sentences)

✓ Huấn luyện hoàn tất
  Kích thước từ vựng (V) : 5,633
  Tổng token             : 71,606
  Số bigram khác nhau    : 33,310


## 4. Tính xác suất câu "Hôm nay trời đẹp lắm"

In [7]:
default_sentence = 'Hôm nay trời đẹp lắm'
target_sentence = read_input_sentence(INPUT_FILE, default_sentence)

prob, log_prob, details = model.sentence_probability(target_sentence)

lines = []
lines.append('=' * 60)
lines.append(f'Câu: "{target_sentence}"')
lines.append('=' * 60)
lines.append(f'\n{"Bigram":<30} {"P(wi|wi-1)":>18}')
lines.append('-' * 50)
for w1, w2, p in details:
    lines.append(f'P({w2:10} | {w1:12}) = {p:.8f}')
lines.append('-' * 50)
lines.append(f'\nXác suất câu  P(s)       =  {prob:.6e}')
lines.append(f'Log xác suất  log P(s)   =  {log_prob:.4f}')
lines.append(f'Perplexity               =  {math.exp(-log_prob / len(details)):.4f}')

result_text = '\n'.join(lines)
print(result_text)

write_output_text(OUTPUT_FILE, result_text)
print(f'\n✓ Đã ghi kết quả vào file: {OUTPUT_FILE.resolve()}')

Câu: "Hôm nay trời đẹp lắm"

Bigram                                 P(wi|wi-1)
--------------------------------------------------
P(hôm        | <s>         ) = 0.00010385
P(nay        | hôm         ) = 0.00017737
P(trời       | nay         ) = 0.00017693
P(đẹp        | trời        ) = 0.00017740
P(lắm        | đẹp         ) = 0.00017721
P(</s>       | lắm         ) = 0.00017749
--------------------------------------------------

Xác suất câu  P(s)       =  1.818514e-23
Log xác suất  log P(s)   =  -52.3614
Perplexity               =  6166.6205

✓ Đã ghi kết quả vào file: E:\Cao học\Xử lý ngôn ngữ tự nhiên\output.txt


## 5. Tính xác suất nhiều câu khác nhau

In [8]:
test_sentences = [
    'Hôm nay trời đẹp lắm',
    'Việt Nam là một đất nước xinh đẹp',
    'Tôi đang học xử lý ngôn ngữ tự nhiên',
    'Hà Nội là thủ đô của Việt Nam',
    'Mô hình ngôn ngữ rất hữu ích',
]

print(f'{"Câu":<45} {"log P(s)":>12} {"Perplexity":>12}')
print('=' * 72)
for sent in test_sentences:
    p, lp, det = model.sentence_probability(sent)
    ppl = math.exp(-lp / len(det)) if det else float('inf')
    print(f'{sent:<45} {lp:>12.4f} {ppl:>12.4f}')

Câu                                               log P(s)   Perplexity
Hôm nay trời đẹp lắm                              -52.3614    6166.6205
Việt Nam là một đất nước xinh đẹp                 -64.0991    1239.0818
Tôi đang học xử lý ngôn ngữ tự nhiên              -74.5546    1729.2812
Hà Nội là thủ đô của Việt Nam                     -66.6862    1651.7312
Mô hình ngôn ngữ rất hữu ích                      -61.3048    2128.3468


## 6. Sinh câu từ mô hình Bigram

In [9]:
random.seed(42)

print('─── Câu sinh ngẫu nhiên (không có từ hạt giống) ─────────────────')
for i in range(5):
    sent = model.generate_sentence(max_len=15, temperature=1.0)
    print(f'  [{i+1}] {sent}')

print()
print('─── Câu sinh với từ hạt giống ───────────────────────────────────')
seeds = ['hôm', 'việt', 'tôi', 'hà', 'mô']
for seed in seeds:
    sent = model.generate_sentence(seed=seed, max_len=15, temperature=0.8)
    print(f'  seed="{seed}" → {sent}')

print()
print('─── Câu sinh với temperature cao (đa dạng hơn) ──────────────────')
for i in range(5):
    sent = model.generate_sentence(max_len=12, temperature=1.5)
    print(f'  [{i+1}] {sent}')

─── Câu sinh ngẫu nhiên (không có từ hạt giống) ─────────────────
  [1] pandanus columnaris tại wikispecies
  [2] the grand est ở hồi hải đảo theo đó an nam
  [3] nhà đầu tiên của avicii được phát thanh kiếm sống
  [4] trentepohlia petulans tại châu thành hình thức am hiểu bởi ferdinand sau họ vào năm
  [5] để tìm ra ở trên bảng xếp hạng nặng nhưng cuối năm 2021

─── Câu sinh với từ hạt giống ───────────────────────────────────
  seed="hôm" → hôm qua đó
  seed="việt" → việt thông qua itunes và herzegovina cho nhiều trong sự tổng hợp thông cáo một xã
  seed="tôi" → tôi có phải ngạc nhiên liệu liên kết ngoài
  seed="hà" → hà nội dung khác
  seed="mô" → mô tả khoa học không xác định được đặt đề cử guiche và là glutamate nhiều

─── Câu sinh với temperature cao (đa dạng hơn) ──────────────────
  [1] morton mô hình thành đến chiết cành hay là lớp tàu ostland
  [2] qua hoạt hóa tiểu đơn như tuyết và 10x cũng đã có
  [3] phục qua địa trung gian tới door to you make me down
  [4] tại nhật
  [5]

## 7. Phân tích mô hình – Top bigram phổ biến nhất

In [10]:
print('Top 20 bigram phổ biến nhất (không tính BOS/EOS):\n')
print(f'  {"Bigram":<25} {"So lan":>8}  {"P(w2|w1)":>12}')
print('  ' + '-' * 50)

all_bigrams = [
    (w1, w2, cnt)
    for w1, w2s in model.bigram_counts.items()
    for w2, cnt in w2s.items()
    if w1 not in (BOS, EOS) and w2 not in (BOS, EOS)
]

all_bigrams.sort(key=lambda x: x[2], reverse=True)
for w1, w2, cnt in all_bigrams[:20]:
    p = model.bigram_prob(w1, w2)
    print(f'  {w1 + " " + w2:<25} {cnt:>8,}  {p:>12.6f}')

Top 20 bigram phổ biến nhất (không tính BOS/EOS):

  Bigram                      So lan      P(w2|w1)
  --------------------------------------------------
  là một                         283      0.043352
  liên quan                      205      0.034506
  liệu liên                      179      0.030790
  quan tới                       178      0.030084
  tham khảo                      132      0.022947
  chú thích                      130      0.022668
  một loài                       116      0.018454
  phát hành                      114      0.019756
  liên kết                       112      0.018928
  đầu tiên                       110      0.018984
  dữ liệu                        107      0.018806
  trong họ                       107      0.017358
  tại wikispecies                106      0.017857
  quá trình                      105      0.018428
  khoa học                       104      0.018255
  kết ngoài                      102      0.017740
  pháp luật                  

In [11]:
print('Top 15 unigram (âm tiết) phổ biến nhất:\n')
for word, cnt in model.unigram_counts.most_common(15):
    if word not in (BOS, EOS):
        print(f'  {word:<15} {cnt:>6,}')

Top 15 unigram (âm tiết) phổ biến nhất:

  và               1,050
  của                923
  là                 918
  được               823
  một                707
  các                599
  trong              589
  có                 557
  năm                522
  ở                  375
  cho                365
  này                361
  tại                359


---
## Tóm tắt kết quả

| Mục | Giá trị |
|-----|--------|
| Corpus | Wikipedia tiếng Việt (200 bài ngẫu nhiên) |
| Mô hình | Bigram + Laplace smoothing |
| Token hoá | Tách âm tiết theo khoảng trắng, chữ thường |
| Dấu đặc biệt | `<s>` (BOS), `</s>` (EOS) |
| Xác suất | P(s) = ∏ P(wᵢ\|wᵢ₋₁) với Laplace smoothing |

**Kết quả xác suất câu "Hôm nay trời đẹp lắm"** được in ở ô 4.  
**Các câu sinh ra** từ mô hình được in ở ô 6.

In [12]:
# ─── 8. Xuất báo cáo đầy đủ ra output.txt ───────────────────────────────────

full_lines = []
full_lines.append('BT01 - BIGRAM LANGUAGE MODEL REPORT')
full_lines.append('=' * 80)

# 1) Cấu hình và thống kê mô hình
full_lines.append('\n[1] MODEL SUMMARY')
full_lines.append('-' * 80)
full_lines.append(f'Số câu sau tiền xử lý: {len(sentences):,}')
full_lines.append(f'Kích thước từ vựng (V): {model.vocab_size:,}')
full_lines.append(f'Tổng token: {sum(model.unigram_counts.values()):,}')
full_lines.append(f'Số bigram khác nhau: {sum(len(v) for v in model.bigram_counts.values()):,}')

# 2) Xác suất câu từ input.txt
full_lines.append('\n[2] SENTENCE PROBABILITY (FROM input.txt)')
full_lines.append('-' * 80)
full_lines.append(f'Câu: "{target_sentence}"')
full_lines.append('')
for w1, w2, p in details:
    full_lines.append(f'P({w2:10} | {w1:12}) = {p:.8f}')
full_lines.append('')
full_lines.append(f'Xác suất câu  P(s)     = {prob:.6e}')
full_lines.append(f'Log xác suất  log P(s) = {log_prob:.4f}')
full_lines.append(f'Perplexity             = {math.exp(-log_prob / len(details)):.4f}')

# 3) Nhiều câu kiểm thử
full_lines.append('\n[3] MULTI-SENTENCE EVALUATION')
full_lines.append('-' * 80)
full_lines.append(f'{"Câu":<50} {"log P(s)":>12} {"Perplexity":>12}')
for sent in test_sentences:
    p, lp, det = model.sentence_probability(sent)
    ppl = math.exp(-lp / len(det)) if det else float('inf')
    full_lines.append(f'{sent:<50} {lp:>12.4f} {ppl:>12.4f}')

# 4) Sinh câu mẫu (giữ ít dòng để file dễ đọc)
full_lines.append('\n[4] GENERATED SENTENCES')
full_lines.append('-' * 80)
random.seed(42)
for i in range(5):
    sent = model.generate_sentence(max_len=15, temperature=1.0)
    full_lines.append(f'Random [{i+1}]: {sent}')

seeds = ['hôm', 'việt', 'tôi', 'hà', 'mô']
for seed in seeds:
    sent = model.generate_sentence(seed=seed, max_len=15, temperature=0.8)
    full_lines.append(f'Seed "{seed}": {sent}')

for i in range(5):
    sent = model.generate_sentence(max_len=12, temperature=1.5)
    full_lines.append(f'HighTemp [{i+1}]: {sent}')

# 5) Top bigram và unigram
full_lines.append('\n[5] TOP 20 BIGRAMS')
full_lines.append('-' * 80)
all_bigrams = [
    (w1, w2, cnt)
    for w1, w2s in model.bigram_counts.items()
    for w2, cnt in w2s.items()
    if w1 not in (BOS, EOS) and w2 not in (BOS, EOS)
]
all_bigrams.sort(key=lambda x: x[2], reverse=True)
for w1, w2, cnt in all_bigrams[:20]:
    p_big = model.bigram_prob(w1, w2)
    full_lines.append(f'{w1 + " " + w2:<30} {cnt:>8,}   P={p_big:.6f}')

full_lines.append('\n[6] TOP 15 UNIGRAMS')
full_lines.append('-' * 80)
count_uni = 0
for word, cnt in model.unigram_counts.most_common(50):
    if word not in (BOS, EOS):
        full_lines.append(f'{word:<20} {cnt:>8,}')
        count_uni += 1
    if count_uni >= 15:
        break

full_report = '\n'.join(full_lines)
write_output_text(OUTPUT_FILE, full_report)

print(f'✓ Đã ghi báo cáo đầy đủ vào: {OUTPUT_FILE.resolve()}')
print(f'✓ Tổng số dòng báo cáo: {len(full_lines):,}')

✓ Đã ghi báo cáo đầy đủ vào: E:\Cao học\Xử lý ngôn ngữ tự nhiên\output.txt
✓ Tổng số dòng báo cáo: 86
